In [2]:
from math import floor 
import pandas as pd
import xml
import xml.etree.ElementTree as ET


In [3]:
# county side information

# More constants: Bounding box for Jefferson county
west_longitude = -85.94712712079293
east_longitude = -85.3443621648922
south_latitude = 37.99712528351634 
north_latitude = 38.38023822809115

min_longitude = west_longitude #
min_latitude = south_latitude #

delta_longitude = east_longitude - west_longitude
delta_latitude = north_latitude - south_latitude

# scale longidude
def scale_longitude(longitude) -> float:
    return (longitude - min_longitude)/delta_longitude

# scale latitude
def scale_latitude(latitude) -> float :
    return (latitude - min_latitude)/delta_latitude

def scale_point(point) -> (float, float):
    longitude, latitude = point 
    longi = (longitude - min_longitude)/delta_longitude
    return longi, (latitude - min_latitude)/delta_latitude


In [33]:
from os import path 
RIDE_DIR = './data/raw/rides'

example_ride = "/Users/bencampbell/code/county_coverage/data/raw/rides/02_03_24.kml"
example_ride_points = "./new/03_input_rides/example_ride"

def read_ride_kml(ride_filename, *, test=False) -> pd.DataFrame:
    root = ET.parse(ride_filename).getroot()
    for item in root.iter():
        if item.tag.endswith('coordinates'):
            break
    coordinate_data = item.text.strip().splitlines()
    if test:
        print(f"file length: {len(coordinate_data)}")
    longitudes = list()
    latitudes = list()
    for coordinate in coordinate_data:
        longitude, latitude, _ = coordinate.split(',')
        # third coordinate element is altitude. Not needed. 
        longitudes.append(float(longitude))
        latitudes.append(float(latitude))
    return pd.DataFrame({"longitude":longitudes, "latitude":latitudes})

lld = read_ride_kml(example_ride, test=False)

cell_width = 14
cell_size = 1 << cell_width
cell_format = f"0{cell_width}b"

def to_cell(value, *, as_string=False):
    out = floor(value * cell_size)
    return out

def get_cells(df):
    longitudes = (df.longitude - min_longitude) / delta_longitude
    longitudes = (longitudes * cell_size).apply(floor)

    latitudes = (df.latitude - min_latitude) / delta_latitude 
    latitudes = (latitudes * cell_size).apply(floor)

    return longitudes.combine(latitudes, lambda x, y:(x,y))

lld['cells'] = get_cells(lld)
lld

,longitude,latitude,cells
0,-85.722366,38.240122,"(6109, 10391)"
1,-85.722364,38.240125,"(6109, 10391)"
2,-85.722353,38.240135,"(6109, 10392)"
3,-85.722332,38.240168,"(6110, 10393)"
4,-85.722286,38.240353,"(6111, 10401)"
...,...,...,...
1394,-85.731065,38.226624,"(5872, 9814)"
1395,-85.731053,38.226602,"(5873, 9813)"
1396,-85.731041,38.226593,"(5873, 9813)"
1397,-85.731026,38.226585,"(5873, 9812)"


In [ ]:
def read_ride_kml(ride_filename, *, test=False) -> pd.DataFrame:
    root = ET.parse(ride_filename).getroot()
    for item in root.iter():
        if item.tag.endswith('coordinates'):
            break
    coordinate_data = item.text.strip().splitlines()
    if test:
        print(f"file length: {len(coordinate_data)}")
    points = list()
    normalized = list()
    for coordinate in coordinate_data:
        longitude, latitude, _ = coordinate.split(',')
        # third coordinate element is altitude. Not needed. 
        longitude = float(longitude)
        latitude = float(latitude)
        points.append((longitude, latitude))
        lo = (longitude - min_longitude)/delta_longitude
        la = (latitude - min_latitude)/delta_latitude
        normalized.append((lo, la))
    return pd.DataFrame({"coordinate":points, "normalized":normalized})

class norms_to_cell:
    def __init__(self, cell_width):
        self.cell_width = cell_width
        self.cell_size = 1 << cell_width
        self.cell_format = f"0{cell_width}b"

    @property
    def cell_formatter(self):
        cell_format = self.cell_format
        return lambda string:format(string, cell_format)


        


        

    
    


read_ride_kml(example_ride)


'02b'

In [ ]:


#lld.scaled_longitude = longitude.apply(scale_longitude)
lld['longitude_scaled'] = lld.longitude.apply(scale_longitude)
lld['latitude_scaled'] = lld.latitude.apply(scale_latitude)

cell_width = 14
cell_size = 1 << cell_width
cell_format = f"0{cell_width}b"

def to_cell(value, *, as_string=False):
    out = floor(value * cell_size)
    return out

lld['longitude_cell'] = lld.longitude_scaled.apply(to_cell)
lld['latitude_cell'] = lld.latitude_scaled.apply(to_cell)

def segment_finder(series):
    memo = None
    starts = list()
    for index, value in enumerate(series):
        if value != memo:
            starts.append(index)
        else:
            pass 
        memo = value
    segments = list()
    start = starts[0]
    for stop in starts[1:]:
        segments.append( slice(start, stop) )
        start = stop
    return segments


    
    
#len(
segment_finder(lld.longitude_cell)
lld.longitude_cell.iloc[slice(10, 12)]

vi = pd.Series(lld.groupby(by=['longitude_cell', 'latitude_cell']).indices)
vii = vi.apply(max).sort_values().index
vi.sort_values(key=lambda series:series.apply(max))



In [138]:
e = lld.longitude
e *= 10
e
lld

,longitude,latitude,longitude_scaled,latitude_scaled,longitude_cell,latitude_cell
0,-85722.366,38.240122,0.372884,0.634269,6109,10391
1,-85722.364,38.240125,0.372887,0.634277,6109,10391
2,-85722.353,38.240135,0.372905,0.634303,6109,10392
3,-85722.332,38.240168,0.372940,0.634389,6110,10393
4,-85722.286,38.240353,0.373016,0.634872,6111,10401
...,...,...,...,...,...,...
1394,-85731.065,38.226624,0.358452,0.599037,5872,9814
1395,-85731.053,38.226602,0.358472,0.598979,5873,9813
1396,-85731.041,38.226593,0.358492,0.598956,5873,9813
1397,-85731.026,38.226585,0.358516,0.598935,5873,9812


In [ ]:


def parseXML(xmlfile):
    # create element tree object
    tree = ET.parse(xmlfile)

    # get root element
    root = tree.getroot()

    # create empty list for news items
    newsitems = []

    # iterate news items
    for item in root.findall('./channel/item'):

        # empty news dictionary
        news = {}

        # iterate child elements of item
        for child in item:

            # special checking for namespace object content:media
            if child.tag == '{https://video.search.yahoo.com/mrss':
                news['media'] = child.attrib['url']
            else:
                news[child.tag] = child.text.encode('utf8')

        # append news dictionary to news items list
        newsitems.append(news)
    
    # return news items list
    return newsitems


In [ ]:

def get_ride_points(filename=example_ride):
    longitudes = list()
    latitudes = list()
    with open("./new/03_input_rides/example_ride") as source:
        for point in source:
            point = point.strip()
            longitude, latitude, altitude = point.split(',')
            longitudes.append(float(longitude))
            latitude.append(float(latitude))
    out = pd.DataFrame()
    out['longitude'] = longitudes
    out['latitude'] = latitude 
    out['scale_longitude'] = 
    out['scale_latitude'] = 
    return out
     
ride = get_ride_points()
ride


In [ ]:
def scale


ride['long_norm'] = (ride.longitude - min_longitude) / delta_longitude
ride['lati_norm'] = (ride.latitude - min_latitude) / delta_latitude 

cell_width = 14
cell_size = 1 << cell_width

def fl(number):
    return f"{floor(number * cell_size):014b}"



ride['long_cell'] = ride.long_norm.apply(fl)
ride['lati_cell'] = ride.lati_norm.apply(fl)

ride


In [ ]:

cell_divisions = 8
cell_width = 2 ** cell_divisions 

def point_to_cell(point) -> (int, int):
    longitude, latitude = point
    longi = (longitude - min_longitude)/delta_longitude
    lati = (latitude - min_latitude)/delta_latitude
    return floor(longi * cell_width), floor(lati * cell_width)

ride.transform({'longitude':(lambda x :x - min_longitude), 'latitude':(lambda x:x-min_latitude)})

long_norm = (ride.longitude - min_longitude) / delta_longitude
lati_norm = (ride.latitude - min_latitude) / delta_longitude

ride['cell_longitude'] = long_norm.apply(lambda x:floor(cell_width * x))
ride['cell_latitude'] = lati_norm.apply(lambda x:floor(cell_width * x))


ride


ModuleNotFoundError: No module named 'geopandas'